# Работа 8. Публичный API и база данных PostgreSQL

Две независимые части: сбор данных из [An API of Ice and Fire](https://anapioficeandfire.com/Documentation)
и запросы к публичной базе [RNAcentral](https://rnacentral.org/help/public-database).

# Часть 1. Работа с API

## Что говорит документация

API отдаёт JSON и не требует авторизации. Существенное для нас:

- **Ресурсы:** `/api/books`, `/api/houses`, `/api/characters`
- **Пагинация:** параметры `page` и `pageSize` (максимум 50 записей на страницу).
  Номер последней страницы в теле ответа не приходит — ссылки на соседние
  страницы лежат в HTTP-заголовке `Link`, и признаком продолжения служит
  `rel="next"` в нём.
- **Фильтры домов:** `region`, `words`, `hasWords`, `hasTitles`, `hasSeats`,
  `hasDiedOut`, `hasAncestralWeapons`.
- **Девиз дома** хранится в поле `words` — например, у Старков это
  «Winter is Coming».

In [1]:
import pandas as pd
import requests

API = "https://anapioficeandfire.com/api"
PAGE_SIZE = 50          # максимум, разрешённый документацией

session = requests.Session()
session.headers.update({"Accept": "application/json"})


def fetch_all(resource: str, **filters) -> list[dict]:
    """Забирает все страницы ресурса и возвращает их одним списком.

    Пагинацию ведём по заголовку Link: пока в нём есть rel="next",
    запрашиваем следующую страницу. Это надёжнее, чем гадать про число
    страниц или крутить цикл, пока ответ не опустеет.
    """
    records: list[dict] = []
    page = 1
    while True:
        response = session.get(f"{API}/{resource}",
                               params={"page": page, "pageSize": PAGE_SIZE, **filters},
                               timeout=30)
        response.raise_for_status()
        batch = response.json()
        records.extend(batch)

        if 'rel="next"' not in response.headers.get("Link", ""):
            return records
        page += 1

## 1.1. Все книги

In [2]:
books = pd.DataFrame(fetch_all("books"))
print(f"книг получено: {len(books)}")
print(f"столбцы: {list(books.columns)}")
books[["name", "authors", "numberOfPages", "publisher", "released"]]

книг получено: 12
столбцы: ['url', 'name', 'isbn', 'authors', 'numberOfPages', 'publisher', 'country', 'mediaType', 'released', 'characters', 'povCharacters']


,name,authors,numberOfPages,publisher,released
0,A Game of Thrones,[George R. R. Martin],694,Bantam Books,1996-08-01T00:00:00
1,A Clash of Kings,[George R. R. Martin],768,Bantam Books,1999-02-02T00:00:00
2,A Storm of Swords,[George R. R. Martin],992,Bantam Books,2000-10-31T00:00:00
3,The Hedge Knight,[George R. R. Martin],164,Dabel Brothers Publishing,2005-03-09T00:00:00
4,A Feast for Crows,[George R. R. Martin],784,Bantam Books,2005-11-08T00:00:00
5,The Sworn Sword,[George R. R. Martin],152,Marvel,2008-06-18T00:00:00
6,The Mystery Knight,[George R. R. Martin],416,Tor Fantasy,2011-03-29T00:00:00
7,A Dance with Dragons,[George R. R. Martin],1040,Bantam Books,2011-07-12T00:00:00
8,The Princess and the Queen,[George R. R. Martin],784,Tor Books,2013-12-03T00:00:00
9,The Rogue Prince,[George R. R. Martin],832,Bantam Books,2014-06-17T00:00:00


## 1.2. Все дома Вестероса

In [3]:
houses = pd.DataFrame(fetch_all("houses"))
print(f"домов получено: {len(houses)}")
print(f"столбцы: {list(houses.columns)}")
houses[["name", "region", "words", "coatOfArms", "founded"]].head(10)

домов получено: 444
столбцы: ['url', 'name', 'region', 'coatOfArms', 'words', 'titles', 'seats', 'currentLord', 'heir', 'overlord', 'founded', 'founder', 'diedOut', 'ancestralWeapons', 'cadetBranches', 'swornMembers']


,name,region,words,coatOfArms,founded
0,House Algood,The Westerlands,,"A golden wreath, on a blue field with a gold b...",
1,House Allyrion of Godsgrace,Dorne,No Foe May Pass,"Gyronny Gules and Sable, a hand couped Or",
2,House Amber,The North,,,
3,House Ambrose,The Reach,Never Resting,"Or, semy of ants gules",
4,House Appleton of Appleton,The Reach,,"Or, an apple tree eradicated proper fructed gu...",
5,House Arryn of Gulltown,The Vale,,,
6,House Arryn of the Eyrie,The Vale,As High as Honor,A sky-blue falcon soaring against a white moon...,Coming of the Andals
7,House Ashford of Ashford,The Reach,Our Sun Shines Bright,"Tenny, a sun in splendour beneath a chevron in...",
8,House Ashwood,The North,,,
9,House Baelish of Harrenhal,The Riverlands,,"A field of silver mockingbirds, on a green fie...",299 AC


## 1.3. Дома, у которых есть девиз

**Про подсказку из задания.** В условии предложено фильтровать параметром
`hasTitles`, но он отбирает дома, у которых заполнены **титулы** (`titles`), а не
девиз. Девиз лежит в поле `words`, и ему соответствует фильтр `hasWords`.
Разница видна по числам: `hasWords=true` даёт 68 домов, `hasTitles=true` — 165.

Основным ответом беру `hasWords`, поскольку задание просит дома с девизом.
Ниже привожу оба варианта, чтобы расхождение было видно, а не спрятано.

In [4]:
houses_with_motto = pd.DataFrame(fetch_all("houses", hasWords="true"))
print(f"домов с девизом (hasWords=true): {len(houses_with_motto)}")

# Проверяем, что фильтр отработал: пустых девизов быть не должно
empty = (houses_with_motto["words"] == "").sum()
print(f"пустых значений в столбце words: {empty}")

houses_with_motto[["name", "region", "words"]].head(10)

домов с девизом (hasWords=true): 68
пустых значений в столбце words: 0


,name,region,words
0,House Allyrion of Godsgrace,Dorne,No Foe May Pass
1,House Ambrose,The Reach,Never Resting
2,House Arryn of the Eyrie,The Vale,As High as Honor
3,House Ashford of Ashford,The Reach,Our Sun Shines Bright
4,House Baratheon of Storm's End,The Stormlands,Ours is the Fury
5,House Beesbury of Honeyholt,The Reach,Beware our Sting
6,House Bolton of the Dreadfort,The North,Our Blades are Sharp
7,House Buckwell of the Antlers,The Crownlands,Pride and Purpose
8,House Bulwer of Blackcrown,The Reach,Death Before Disgrace
9,House Caron of Nightsong,The Stormlands,No Song so Sweet


In [5]:
# Тот же запрос буквально по подсказке — для сравнения
houses_with_titles = pd.DataFrame(fetch_all("houses", hasTitles="true"))

print(f"hasWords=true  (есть девиз):  {len(houses_with_motto):>3} домов")
print(f"hasTitles=true (есть титулы): {len(houses_with_titles):>3} домов")
print(f"всего домов в API:            {len(houses):>3}")

# Сколько домов из выборки по титулам на самом деле без девиза
without_motto = (houses_with_titles["words"] == "").sum()
print(f"\nиз домов с титулами не имеют девиза: {without_motto}")

hasWords=true  (есть девиз):   68 домов
hasTitles=true (есть титулы): 165 домов
всего домов в API:            444

из домов с титулами не имеют девиза: 124


# Часть 2. Работа с базой данных

## 2.1. Установка библиотеки

```
pip install psycopg2-binary
```

Ставится именно `psycopg2-binary`, а не `psycopg2`: сборка идёт уже
скомпилированной, без необходимости иметь на машине компилятор и заголовки
PostgreSQL. Имя пакета при импорте одинаковое — `psycopg2`.

## 2.2. Подключение к базе

Параметры подключения взяты со страницы публичной базы RNAcentral. Доступ
открытый и **только на чтение** — база опубликована специально для обучения
и внешних запросов.

In [6]:
import time

import psycopg2

DB_PARAMS = {
    "host": "hh-pgsql-public.ebi.ac.uk",
    "port": 5432,
    "dbname": "pfmegrnargs",
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "connect_timeout": 20,
}


def connect(attempts: int = 5) -> psycopg2.extensions.connection:
    """Подключается к базе, повторяя попытку при отказе.

    Сервер общий для всех желающих, и в загруженный момент он отвечает
    «remaining connection slots are reserved for roles with the SUPERUSER
    attribute» — свободных слотов подключения нет. Это временное состояние,
    поэтому повторяем попытку с нарастающей паузой.
    """
    for attempt in range(1, attempts + 1):
        try:
            return psycopg2.connect(**DB_PARAMS)
        except psycopg2.OperationalError as error:
            if attempt == attempts:
                raise
            print(f"попытка {attempt}/{attempts} не удалась: "
                  f"{str(error).strip().splitlines()[0]}")
            time.sleep(5 * attempt)


connection = connect()
print("подключение установлено")
print("сервер:", connection.get_dsn_parameters()["host"])
print("база:  ", connection.get_dsn_parameters()["dbname"])

подключение установлено
сервер: hh-pgsql-public.ebi.ac.uk
база:   pfmegrnargs


## 2.3. Десять строк из таблицы `rnc_database`

In [7]:
cursor = connection.cursor()
cursor.execute("SELECT * FROM rnc_database LIMIT 10")
rows = cursor.fetchall()

print(f"получено строк: {len(rows)}")
print(f"значений в строке: {len(rows[0])}")
rows[0][:6]

получено строк: 10
значений в строке: 19


(21, datetime.datetime(2017, 5, 2, 0, 0), 'RNACEN', 'NONCODE', 146, 'NONCODE')

## 2.4. Сохранение результата в DataFrame

In [8]:
# cursor.description хранит метаданные последнего запроса,
# первый элемент каждой записи — имя столбца. Без этого DataFrame
# получил бы безымянные колонки 0, 1, 2...
column_names = [column[0] for column in cursor.description]

databases = pd.DataFrame(rows, columns=column_names)
print(f"строк: {databases.shape[0]}, столбцов: {databases.shape[1]}")
databases[["id", "display_name", "num_sequences", "alive"]]

строк: 10, столбцов: 19


,id,display_name,num_sequences,alive
0,21,NONCODE,234669,Y
1,5,VEGA,0,N
2,26,GENCODE,47677,N
3,1,ENA,12086180,Y
4,14,TAIR,4406,Y
5,9,RefSeq,120355,Y
6,41,GeneCards,425357,Y
7,10,RDP,4779,Y
8,20,LNCipedia,126876,Y
9,15,WormBase,25550,Y


## 2.5. Отдельные столбцы: `display_name`, `num_sequences`, `num_organisms`, `url`

In [9]:
cursor.execute("""
    SELECT display_name, num_sequences, num_organisms, url
    FROM rnc_database
    LIMIT 10
""")
selected = pd.DataFrame(cursor.fetchall(),
                        columns=[column[0] for column in cursor.description])
selected

,display_name,num_sequences,num_organisms,url
0,NONCODE,234669,7,http://www.noncode.org/
1,VEGA,0,0,http://vega.sanger.ac.uk/
2,GENCODE,47677,2,http://gencodegenes.org/
3,ENA,12086180,814855,https://www.ebi.ac.uk/ena/browser/
4,TAIR,4406,1,http://www.arabidopsis.org/
5,RefSeq,120355,22524,http://www.ncbi.nlm.nih.gov/refseq/
6,GeneCards,425357,1,https://www.genecards.org/
7,RDP,4779,2487,http://rdp.cme.msu.edu/
8,LNCipedia,126876,1,http://www.lncipedia.org/
9,WormBase,25550,1,http://www.wormbase.org/


In [10]:
# Закрываем курсор и соединение: они держат сетевой сокет и место
# в пуле подключений на стороне сервера
cursor.close()
connection.close()
print("соединение закрыто:", connection.closed == 1)

соединение закрыто: True


## Итог

**API.** Получены 12 книг, 444 дома и 68 домов с девизом. Пагинация ведётся по
заголовку `Link`, а не по счётчику страниц: API не сообщает их общее число в теле
ответа.

**База данных.** Через `psycopg2` выполнены два запроса к таблице `rnc_database`
публичной базы RNAcentral — полная выборка из 10 строк (19 столбцов) и выборка
четырёх нужных столбцов. Имена столбцов для `DataFrame` взяты из
`cursor.description`, поскольку `fetchall()` возвращает только значения.

**Отличие от подсказки.** Для домов с девизом использован фильтр `hasWords`,
а не `hasTitles` из условия: девиз хранится в поле `words`, титулы — в `titles`,
и выборки отличаются более чем вдвое (68 против 165).